In [ ]:
#pip install h5py nibabel pandas glob2 regex
import os
import sys
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from tqdm import tqdm
from itertools import product
import scipy
import torch
import torch.nn as nn
os.environ["CUDA_VISIBLE_DEVICES"]= '1' #, this way I would choose GPU 3 to do the work
import scipy.io as sio
sys.path.append(os.path.abspath(".."))
sys.path.append(os.path.abspath("../src"))

from walinet.inference.inference import * 
from walinet.inference.fid_inference import infer_fid
from walinet.training_data.lipid_removal import * 
from walinet.visualization.EvaluateModel import * 

# Run-Config
Scanner = "7T"

sub = "Measurement26_594"

add_water = False

if Scanner == "3T":
    Vol = "B0corrected_wo_LipidMask/PRISMA_Vienna/Vol01_WB/Res50x50"
    version = 'v_1.1'
    model_1 = '3T_Final_32_Channels'
    #model_2 = '3T_50x50_Unet_1.1_6Layer'

elif Scanner == "7T":
    Vol = f"NoB0Correction/{sub}" #f"B0corrected_wo_LipidMask/{sub}" #"NoB0Correction/MS_{MS}"#"NoB0Correction/MS_180" #"B0corrected_wo_LipidMask/Vol5"
    version = 'v_1.0' # data version
    model_1 = '7T_Final' #'UNet_ablation_parameter_matched'

elif Scanner == "10.5T":
    Vol = f"{sub}" #f"B0corrected_wo_LipidMask/{sub}" #"NoB0Correction/MS_{MS}"#"NoB0Correction/MS_180" #"B0corrected_wo_LipidMask/Vol5"
    version = 'v_1.0' # data version
    model_1 = '10p5T_provisional' #'UNet_ablation_parameter_matched'
   # model_2 = 'UNet_ablation_parameter_matched'#"7T_Unet_1.1_6Layers"#'7T_Ynet_Final_proj_en_1.1'    #'Final_B0correction_In_Brain', "Vienna_5DataSets_1.1_7Layers"
     #max_abs' #projection_enery'

In [ ]:

#model_dir = os.path.abspath(f'../models/{model}')
#src_dir   = os.path.join(model_dir, 'src')
#sys.path.insert(0, model_dir)
#sys.path.insert(0, src_dir)

## Load spectra
T1 = np.load(f'../data/{Scanner}/{Vol}/OriginalData/magnitude.npy')


FID = np.load(f'../data/{Scanner}/{Vol}/OriginalData/data.npy')
spectra = np.fft.fftshift(np.fft.fft(FID, axis=-1), axes=-1)

brainmask = np.load(f'../data/{Scanner}/{Vol}/masks/brain_mask.npy')
skmask = np.load(f'../data/{Scanner}/{Vol}/masks/lipid_mask.npy')
headmask = brainmask + skmask

if add_water:
    from pathlib import Path
    from walinet.parameter_calibration.load_data import (
        load_subject_simulation_water_fids,
    )

    subject_dir = Path(f'../data/{Scanner}/{Vol}').resolve()
    water, water_brainmask = load_subject_simulation_water_fids(
        subject_dir,
        resource_relative_path=(
            'TrainData/SimulationResources_water_lipid_v1.h5'
        ),
        verify_external_mask=True,
    )
    if water.shape != FID.shape:
        raise ValueError(
            'Water and subject FIDs have incompatible shapes:\n'
            f'  water: {water.shape}\n'
            f'  FID:   {FID.shape}\n'
            f'  subject: {subject_dir}'
        )
    print(f'Loaded water FIDs from {subject_dir.name}: {water.shape}')

FID_Length = FID.shape[-1]

# This evaluation uses an operator-free U-Net. No lipid projection
# operator is required or loaded from the simulation resources.
lipid_operator = None
print("Lipid projection disabled; continuing without an operator.")


In [ ]:
# # Pure network-forward benchmark (model loading and preprocessing are not timed).
# import time
# from pathlib import Path

# from walinet.inference.inference import (
#     _load_checkpoint_state_dict,
#     _load_model_and_params,
# )
# from walinet.inference.fid_inference import (
#     _fid_to_spectrum,
#     _load_acquisition_info,
#     _prepare_fid_length,
# )

# BENCHMARK_BATCH_SIZE = 200
# BENCHMARK_REPEATS = 1  # Increase for more stable timing statistics.
# BENCHMARK_CHECKPOINT = 'model_last.pt'

# benchmark_model_dir = Path(f'../models/{model_1}').resolve()
# model_cls, benchmark_params, benchmark_architecture, loaded_model_dir = (
#     _load_model_and_params(
#         exp=benchmark_model_dir.name,
#         model_root=benchmark_model_dir.parent,
#         architecture='auto',
#     )
# )
# if benchmark_architecture != 'unet':
#     raise NotImplementedError(
#         'This pure-forward benchmark currently targets the operator-free WALRUS U-Net.'
#     )

# # Prepare and normalize all selected spectra once, outside the timed section.
# benchmark_fid = np.asarray(FID, dtype=np.complex64)
# benchmark_fid = _prepare_fid_length(
#     benchmark_fid, _load_acquisition_info(loaded_model_dir)
# )
# benchmark_spectra = _fid_to_spectrum(benchmark_fid).reshape(-1, benchmark_fid.shape[-1])
# benchmark_indices = np.flatnonzero(np.asarray(headmask).reshape(-1) > 0)
# benchmark_spectra = benchmark_spectra[benchmark_indices]
# benchmark_spectra = benchmark_spectra[
#     np.isfinite(benchmark_spectra).all(axis=1)
# ]
# benchmark_norm = np.maximum(
#     np.max(np.abs(benchmark_spectra), axis=1, keepdims=True), 1e-8
# )
# benchmark_normalized = benchmark_spectra / benchmark_norm
# benchmark_network_input = np.stack(
#     (benchmark_normalized.real, benchmark_normalized.imag), axis=1
# ).astype(np.float32, copy=False)

# def _loaded_benchmark_model(benchmark_device):
#     model = model_cls(
#         nLayers=int(benchmark_params['nLayers']),
#         nFilters=int(benchmark_params['nFilters']),
#         dropout=float(benchmark_params.get('dropout', 0.0)),
#         in_channels=int(benchmark_params['in_channels']),
#         out_channels=int(benchmark_params['out_channels']),
#     )
#     checkpoint_path = loaded_model_dir / BENCHMARK_CHECKPOINT
#     model.load_state_dict(
#         _load_checkpoint_state_dict(checkpoint_path, benchmark_device)
#     )
#     return model.to(benchmark_device).eval()

# def _time_pure_forward(device_name):
#     benchmark_device = torch.device(device_name)
#     model = _loaded_benchmark_model(benchmark_device)  # deliberately before timer

#     # Warm up kernels/caches with one batch; also excluded from the timer.
#     warmup = torch.as_tensor(
#         benchmark_network_input[:BENCHMARK_BATCH_SIZE],
#         dtype=torch.float32,
#         device=benchmark_device,
#     )
#     with torch.inference_mode():
#         _ = model(warmup)
#     if benchmark_device.type == 'cuda':
#         torch.cuda.synchronize(benchmark_device)
#     del warmup

#     repeat_times = []
#     with torch.inference_mode():
#         for _ in range(BENCHMARK_REPEATS):
#             forward_seconds = 0.0
#             for start in range(0, len(benchmark_network_input), BENCHMARK_BATCH_SIZE):
#                 # Tensor creation and host-to-device transfer are outside the timer.
#                 batch = torch.as_tensor(
#                     benchmark_network_input[start:start + BENCHMARK_BATCH_SIZE],
#                     dtype=torch.float32,
#                     device=benchmark_device,
#                 )
#                 if benchmark_device.type == 'cuda':
#                     torch.cuda.synchronize(benchmark_device)
#                 started = time.perf_counter()
#                 output = model(batch)
#                 if benchmark_device.type == 'cuda':
#                     torch.cuda.synchronize(benchmark_device)
#                 forward_seconds += time.perf_counter() - started
#                 del batch, output
#             repeat_times.append(forward_seconds)

#     median_seconds = float(np.median(repeat_times))
#     print(
#         f'{benchmark_device}: pure forward = {median_seconds:.3f} s for '
#         f'{len(benchmark_network_input):,} spectra '
#         f'({1e3 * median_seconds / len(benchmark_network_input):.4f} ms/spectrum; '
#         f'batch size {BENCHMARK_BATCH_SIZE}, median of {BENCHMARK_REPEATS})'
#     )
#     del model
#     if benchmark_device.type == 'cuda':
#         torch.cuda.empty_cache()
#     return median_seconds

# print(
#     'Timing scope: model forward only; excludes model/checkpoint loading, FFT, '
#     'normalization, masking, device transfer, and postprocessing.'
# )
# if not torch.cuda.is_available():
#     raise RuntimeError('CUDA is unavailable; the GPU-only benchmark cannot run.')

# # CUDA_VISIBLE_DEVICES='1' above exposes physical GPU 1 as logical cuda:0.
# benchmark_gpu = 'cuda:0'
# print(
#     f'Benchmark device: {benchmark_gpu} '
#     f'(physical GPU selected by CUDA_VISIBLE_DEVICES={os.environ.get("CUDA_VISIBLE_DEVICES")}; '
#     f'{torch.cuda.get_device_name(torch.device(benchmark_gpu))})'
# )
# pure_inference_time_gpu = _time_pure_forward(benchmark_gpu)
# pure_inference_time_cpu = _time_pure_forward('cpu')
# print(f'GPU speed-up: {pure_inference_time_cpu / pure_inference_time_gpu:.2f}x')


In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
clean_fid_model_1 = infer_fid(
    fid=FID,
    model_dir=f"../models/{model_1}",
    fid_axis=-1,
    headmask=headmask,
    lipid_projection_operator=lipid_operator,
    checkpoint="model_last.pt",
    device=device,
)
out_model_1 = np.fft.fftshift(
    np.fft.fft(clean_fid_model_1, axis=-1),
    axes=-1,
)


# compute second model if specified
if "model_2" in globals() and model_2 not in [None, ""]:
    clean_fid_model_2 = infer_fid(
        fid=FID,
        model_dir=f"../models/{model_2}",
        fid_axis=-1,
        headmask=headmask,
        lipid_projection_operator=lipid_operator,
        checkpoint="model_last.pt",
        device=device,
    )
    out_model_2 = np.fft.fftshift(
        np.fft.fft(clean_fid_model_2, axis=-1),
        axes=-1,
    )

In [ ]:
# from pathlib import Path

# subject_original_data = Path(
#     f'../data/{Scanner}/{Vol}/OriginalData'
# ).resolve()


# # WALINET output: frequency domain (X, Y, Z, T).
# nuisance_free_fids = np.fft.ifft(
#     np.fft.ifftshift(out_model_1, axes=-1),
#     axis=-1,
# )[..., :FID_Length]

# if nuisance_free_fids.shape != FID.shape:
#     raise ValueError(
#         f'Predicted and original FIDs have different shapes: '
#         f'{nuisance_free_fids.shape} vs. {FID.shape}'
#     )

# if add_water:
#     if water.shape != nuisance_free_fids.shape:
#         raise ValueError(
#             f'Water and predicted FIDs have different shapes: '
#             f'{water.shape} vs. {nuisance_free_fids.shape}'
#         )
#     nuisance_free_fids = nuisance_free_fids + water

# # Restore the desired spatial orientation by swapping only X and Y.
# data_for_reco = nuisance_free_fids.astype(np.complex64, copy=False)

# output_path = (f'{sub}_{model_1}.mat'
# )

# sio.savemat(
#     output_path,
#     {
#         'Data': data_for_reco,
#         'mask': brainmask,
#     },
# )

# print('WALINET FIDs:', nuisance_free_fids.shape)
# print('Reco Data:', data_for_reco.shape)
# print('Saved:', output_path)

In [ ]:
# bring back to FID domain for saving the data
# Prediction_time = np.fft.ifft(np.fft.ifftshift(Data_LipidRemoved_rrrf, axes=-1), axis=-1)

# np.save(f'../data/{Vol}/OriginalData/data_after_walinet.npy', Prediction_time)

# print(Prediction_time.shape)

In [ ]:
# # Path to your file
# file_path = f'../data/{Vol}/OriginalData/IsolatedWater.npy'

# # Open the file in read mode
# water = np.load(file_path)

# z,t = 17, 5

# #Slice = np.abs(water)[:,:,z,t]

# #plt.imshow(Slice)

# combined = Prediction_time + water

# combined_swapped = np.swapaxes(combined, 0,1)

# print(combined_swapped.shape)
# sio.savemat('Vol01_WB_Thick_before_Walinet.mat', {'Data': combined_swapped})


In [ ]:
%matplotlib widget

BeforeWALINET = np.real(spectra)
# Nuisance = np.real(spectra-Data_LipidRemoved_rrrf)
# NuisanceRemoved = np.real(Data_LipidRemoved_rrrf)

# Nuisance = np.real(spectra-out_ynet)
# NuisanceRemoved = np.real(out_ynet)

if Scanner == "3T":
    ppm = make_ppm_axis(
        vec_size=288,
        dwelltime=1063800,
        larmor_freq=123231706,
        ppm_ref=4.7,
        ref_freq_hz=0.0,
        dwelltime_unit="ns",
    )

    metabolite_lines = {   # TwoHG, typischer Hauptbereich ca. 4.02 ppm
        "water": 4.7,
        "Cr": 3.02, 
        "GABA": 2.28, 
        "GPC": 3.23, 
        "GSH": 2.95, 
        "Gln": 2.45, 
        "Glu": 2.35, 
        "Ins": 3.56,   # myo-Inositol
        "NAA": 2.01,
        "NAAG": 2.04,
        "PCh": 3.22,
        "PCr": 3.93,
    }

if Scanner == "7T":
    ppm = make_ppm_axis(
        vec_size=FID_Length,#840,
        dwelltime=360000,
        larmor_freq=297222876,
        ppm_ref=4.7,
        ref_freq_hz=0.0,
        dwelltime_unit="ns",
    )

    metabolite_lines = {   # TwoHG, typischer Hauptbereich ca. 4.02 ppm
        "water": 4.7,
        "Glc": 3.8,
        "Asp": 2.8,
        "Cr": 3.02, 
        "GABA": 2.28, 
        "GPC": 3.23, 
        "GSH": 2.95, 
        "Gln": 2.45, 
        "Glu": 2.35, 
        "Ins": 3.56,   # myo-Inositol
        "NAA": 2.01,
        "NAAG": 2.04,
        "PCh": 3.22,
        "PCr": 3.93,
        "Scyllo": 3.35,
        "Tau": 3.42,
        "2HG": 2.25,
        "Gly": 3.55,
    }

if Scanner == "10.5T":
    ppm = make_ppm_axis(
        vec_size=FID_Length,#840,
        dwelltime=360000,
        larmor_freq=447053391,
        ppm_ref=4.7,
        ref_freq_hz=0.0,
        dwelltime_unit="ns",
    )

    metabolite_lines = {   # TwoHG, typischer Hauptbereich ca. 4.02 ppm
        "water": 4.7,
        "Glc": 3.8,
        "Asp": 2.8,
        "Cr": 3.02, 
        "GABA": 2.28, 
        "GPC": 3.23, 
        "GSH": 2.95, 
        "Gln": 2.45, 
        "Glu": 2.35, 
        "Ins": 3.56,   # myo-Inositol
        "NAA": 2.01,
        "NAAG": 2.04,
        "PCh": 3.22,
        "PCr": 3.93,
        "Scyllo": 3.35,
        "Tau": 3.42,
        "2HG": 2.25,
        "Gly": 3.55,
    }

In [ ]:
#water = np.load(f'../data/{Scanner}/{Vol}/TrainData/IsolatedWater_v_1.1_final.npy')
#out_model_1 = np.fft.fftshift(np.fft.fft(water, axis=-1), axes=-1)

Nuisance_model_1 = np.real(spectra - out_model_1)
Model_1 = np.real(out_model_1)   #np.fft.ifft(np.fft.ifftshift(water, axes=-1), axis=-1)

has_model_2 = (
    "model_2" in globals()
    and model_2 is not None
    and model_2 != ""
    and "out_model_2" in globals()
    and out_model_2 is not None
)

if has_model_2:
    Nuisance_model_2 = np.real(spectra - out_model_2)
    Model_2 = np.real(out_model_2)


def vol_real(vol):
    return lambda x, y, z: np.real(vol[x, y, z, :])


spectrum_funcs = [
    make_spec_func(BeforeWALINET),
    make_spec_func(Nuisance_model_1),
    make_spec_func(Model_1),
]

labels = [
    "Input / WALINET comparison",
    model_1,
]

overlay_labels = [
    "Input",
    model_1,
]

extra_spectrum_funcs = None
extra_labels = None

if has_model_2:
    extra_spectrum_funcs = [
        vol_real(Nuisance_model_2),
        vol_real(Model_2),
    ]

    extra_labels = [
        model_2,
        model_2,
    ]


interactive_spectra_viewer_multi(
    image_volume=T1,
    spectrum_funcs=spectrum_funcs,
    labels=labels,
    overlay_labels=overlay_labels,
    extra_spectrum_funcs=extra_spectrum_funcs,
    extra_labels=extra_labels,
    x_axis=ppm,
    x_label="ppm",
    ppm_min=4,
    ppm_max=0,
    compact=True,
    metabolite_lines=metabolite_lines,
)

In [ ]:
# asas

# Validation set inference

In [ ]:
# res_1 = runNNOnTrainDataH5(
#     device=device,
#     exp=model_1,
#     h5_path=f"../data/{Scanner}/{Vol}/TrainData/TrainData_{version}.h5",
#     checkpoint="model_best.pt",
#     return_all=True,
# )

# res_2 = None

# if (
#     "model_2" in globals()
#     and model_2 is not None
#     and model_2 != ""
# ):
#     res_2 = runNNOnTrainDataH5(
#         device=device,
#         exp=model_2,
#         h5_path=f"../data/{Scanner}/{Vol}/TrainData/TrainData_{version}.h5",
#         checkpoint="model_best.pt",
#         return_all=True,
#     )

In [ ]:
# plot_train_inference_grid(
#     res_1,
#     res_extra=res_2,
#     indices=range(0, 50),
#     x_axis=ppm,
#     x_label="ppm",
#     ppm_min=0,
#     ppm_max=4,
#     label=model_1,
#     extra_label=model_2 if res_2 is not None else "",
# )

In [ ]:
# print(model_1, compute_normalized_nuisance_mse(res_1, "max_abs"))

# if res_2 is not None:
#     print(model_2, compute_normalized_nuisance_mse(res_2, "max_abs"))

In [ ]:
# 

In [ ]:
# # Compare the loaded WALINET configurations on newly generated simulations.
# from pathlib import Path
# from walinet.training_data.build_simulation_system import build_simulation_system
# from walinet.visualization.VisualizeSim import plot_simulation_model_comparison

# N_SIMULATIONS = 100
# SIMULATION_SEED = 123456

# if Scanner not in {"3T", "7T"}:
#     raise ValueError(f"No simulation configuration defined for Scanner={Scanner!r}.")

# simulation_training_config = (
#     Path("../configs/Training") / f"VisulizationNotebook_{Scanner}.yaml"
# ).resolve()
# simulation_system = build_simulation_system(simulation_training_config)
# simulation_cfg = simulation_system.simulation_config
# simulation_generator = torch.Generator(device=simulation_system.device)
# simulation_generator.manual_seed(SIMULATION_SEED)

# simulation_batch = simulation_system.train_simulator.simulate(
#     batch_size=N_SIMULATIONS,
#     generator=simulation_generator,
# )

# sim_input = simulation_batch.normalized_input_spectra.detach().cpu().numpy()
# sim_nuisance_gt = simulation_batch.normalized_target_spectra.detach().cpu().numpy()
# sim_metabolites_gt = sim_input - sim_nuisance_gt

# def infer_simulated_metabolites(model_name):
#     # runNNLipRemoval2 expects a four-dimensional spectral volume.
#     simulated_volume = sim_input[:, None, None, :]
#     clean_volume = runNNLipRemoval2(
#         device=device,
#         exp=model_name,
#         spectra=simulated_volume,
#         headmask=np.ones(simulated_volume.shape[:-1], dtype=bool),
#         batch_size=N_SIMULATIONS,
#         normalization="auto",
#         architecture="auto",
#         model_root="../models",
#         checkpoint="model_last.pt",
#     )
#     return clean_volume[:, 0, 0, :]

# simulation_metabolite_predictions = {
#     model_1: infer_simulated_metabolites(model_1),
# }

# if "model_2" in globals() and model_2 not in (None, ""):
#     simulation_metabolite_predictions[model_2] = infer_simulated_metabolites(model_2)

# print(f"Generated {N_SIMULATIONS} simulations with {sim_input.shape[-1]} points.")
# print("Models:", list(simulation_metabolite_predictions))


In [ ]:
# # Select the half-open simulation index range [start, stop).
# plot_simulation_model_comparison(
#     total_spectra=sim_input,
#     nuisance_ground_truth=sim_nuisance_gt,
#     metabolite_predictions=simulation_metabolite_predictions,
#     simulation_cfg=simulation_cfg,
#     start=20,
#     stop=70,
#     component="real",
#     ppm_min=1.0,
#     ppm_max=4.0,
# )
